In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# ==========================================================
# SETTINGS
# ==========================================================

input_dir = '.'   # directory with CBC excel files
tracker_file = "Experiment Tracker.xlsx"

output_dir = "CBC_Plots"
os.makedirs(output_dir, exist_ok=True)

plt.rcParams['axes.axisbelow'] = True

# ==========================================================
# TIME CONTROL
# ==========================================================

order_of_time_points = [0, 30, 60, 85, 120, 240]
X_TICK_INTERVAL = 30

# ==========================================================
# VISUAL SETTINGS
# ==========================================================

AXIS_TICK_SIZE = 16
LABEL_SIZE = 18
LEGEND_SIZE = 14

color_palette = [
    'black', 'blue', 'red',
    'purple', 'orange',
    'cyan', 'magenta'
]

occlusion_label_map = {
    'No Occlusion': 'No REBOA',
    'Partial Occlusion': 'p-REBOA',
    'Full Occlusion': 'f-REBOA'
}

occlusion_order = [
    'No Occlusion',
    'Partial Occlusion',
    'Full Occlusion'
]

# ==========================================================
# CBC METRIC LABELS
# ==========================================================

metric_name_map = {
    "wbc": "WBC (10³/µL)",
    "rbc": "RBC (10⁶/µL)",
    "hgb": "Hemoglobin (g/dL)",
    "hct": "Hematocrit (%)",
    "plt": "Platelets (10³/µL)"
}

# ==========================================================
# PARSE NAME COLUMN
# ==========================================================

# def parse_name(value):
#     try:
#         parts = str(value).split('-')
#         subject = '-'.join(parts[:2])   # ERNE-61
#         time_point = parts[2]          # PRE / T30
#         return subject, time_point
#     except:
#         return None, None

def parse_name(value):
    try:
        parts = str(value).split('-')

        # Convert ERNE-61 → ERNE61
        subject = parts[0] + parts[1]

        time_point = parts[2]

        return subject, time_point
    except:
        return None, None
    
    

def convert_time(tp):
    if tp == 'PRE':
        return 0
    if isinstance(tp, str) and tp.startswith('T'):
        try:
            return float(tp.replace('T', ''))
        except:
            return None
    return None

# ==========================================================
# LOAD TRACKER
# ==========================================================

tracker = pd.read_excel(
    tracker_file,
    sheet_name=['T30 categorization', 'T60 categorization']
)

group_10 = set(tracker['T30 categorization']['Group_10'].dropna())
group_20 = set(tracker['T30 categorization']['Group_20'].dropna())
group_30 = set(tracker['T30 categorization']['Group_30'].dropna())

full_occlusion = set(tracker['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(tracker['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(tracker['T60 categorization']['No_Occlusion'].dropna())

# ==========================================================
# LOAD + CLEAN ALL CBC FILES
# ==========================================================

all_data = []

for file in os.listdir(input_dir):
    if file.endswith('.xlsx') and not file.startswith('~$') and file != tracker_file:

        print(f"Processing {file}")
        df = pd.read_excel(os.path.join(input_dir, file))

        if 'name' not in df.columns:
            continue

        # Split name column
        parsed = df['name'].apply(parse_name)
        df[['Subject', 'Time Point']] = pd.DataFrame(parsed.tolist(), index=df.index)

        # Convert time
        df['Time Point'] = df['Time Point'].apply(convert_time)

        # Drop bad rows
        df = df.dropna(subset=['Subject', 'Time Point'])

        all_data.append(df)

# Combine all files
df = pd.concat(all_data, ignore_index=True)

# ==========================================================
# APPLY TRACKER LOGIC
# ==========================================================

def assign_hemorrhage(subj):
    if subj in group_10:
        return 10
    elif subj in group_20:
        return 20
    elif subj in group_30:
        return 30
    return np.nan

def assign_occlusion(subj):
    if subj in full_occlusion:
        return 'Full Occlusion'
    elif subj in partial_occlusion:
        return 'Partial Occlusion'
    elif subj in no_occlusion:
        return 'No Occlusion'
    return np.nan

df['Hemorrhage Level'] = df['Subject'].apply(assign_hemorrhage)
df['Occlusion Group'] = df['Subject'].apply(assign_occlusion)

# Remove subjects not in tracker
df = df.dropna(subset=['Hemorrhage Level', 'Occlusion Group'])

# ==========================================================
# FILTER TIME POINTS
# ==========================================================

df = df[df['Time Point'].isin(order_of_time_points)]
df = df.sort_values('Time Point')




# ==========================================================
# EXPORT FINAL CLEANED DATA (FOR DEBUGGING)
# ==========================================================

debug_output_file = "CBC_combined_debug.xlsx"

df.to_excel(debug_output_file, index=False)

print(f"\n✅ Debug file saved: {debug_output_file}")
print("Shape:", df.shape)



# ==========================================================
# IDENTIFY METRICS
# ==========================================================

exclude_cols = [
    'Subject',
    'Time Point',
    'Hemorrhage Level',
    'Occlusion Group',
    'name'
]

numeric_columns = df.select_dtypes(include=['number']).columns
metrics = [col for col in numeric_columns if col not in exclude_cols]

# ==========================================================
# PLOTTING FUNCTION
# ==========================================================

def create_plots_for_metric(metric):

    hemorrhage_levels = [10, 20, 30]
    fig, axes = plt.subplots(1, 3, figsize=(30, 8), sharey=True)

    handles, labels = [], []

    for ax, hemorrhage_level in zip(axes, hemorrhage_levels):

        level_df = df[df['Hemorrhage Level'] == hemorrhage_level]

        if level_df.empty:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage - No Data",
                         fontsize=LABEL_SIZE)
            continue

        mean_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].mean()

        std_df = level_df.groupby(
            ['Time Point', 'Occlusion Group'],
            as_index=False
        )[metric].std()

        for i, occlusion in enumerate(occlusion_order):

            if occlusion not in mean_df['Occlusion Group'].unique():
                continue

            mean_subset = mean_df[mean_df['Occlusion Group'] == occlusion]
            std_subset = std_df[std_df['Occlusion Group'] == occlusion]

            color = color_palette[i % len(color_palette)]
            legend_label = occlusion_label_map.get(occlusion, occlusion)

            x_vals = mean_subset['Time Point']

            line, = ax.plot(
                x_vals,
                mean_subset[metric],
                marker='o',
                markersize=8,
                linewidth=2,
                label=legend_label,
                color=color
            )

            ax.fill_between(
                x_vals,
                mean_subset[metric] - std_subset[metric],
                mean_subset[metric] + std_subset[metric],
                alpha=0.2,
                color=color
            )

            if legend_label not in labels:
                handles.append(line)
                labels.append(legend_label)

        ax.set_title(f"{hemorrhage_level}% Hemorrhage",
                     fontsize=LABEL_SIZE)

        ax.set_xlabel("Time (mins)", fontsize=LABEL_SIZE)

        ax.set_xticks(
            np.arange(
                min(order_of_time_points),
                max(order_of_time_points) + 1,
                X_TICK_INTERVAL
            )
        )

        ax.minorticks_on()
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))

        ax.grid(True, which='major', linestyle='-', linewidth=0.7, alpha=0.6)
        ax.grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.4)

        ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

    y_label = metric_name_map.get(metric, metric)
    axes[0].set_ylabel(y_label, fontsize=LABEL_SIZE)

    plt.tight_layout()

    plt.savefig(
        os.path.join(output_dir, f"{metric}.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

# ==========================================================
# RUN ALL METRICS
# ==========================================================

for metric in metrics:
    print(f"Plotting {metric}")
    create_plots_for_metric(metric)

print("✅ DONE: CBC processing + plotting complete.")

Processing CBC ERNE-66.xlsx
Processing CBC ERNE-31.xlsx
Processing CBC ERNE-27.xlsx
Processing CBC ERNE-70.xlsx
Processing CBC ERNE-07.xlsx
Processing CBC ERNE-50.xlsx
Processing CBC ERNE-46.xlsx
Processing CBC ERNE-11.xlsx
Processing CBC ERNE-10.xlsx
Processing CBC ERNE-47.xlsx
Processing CBC ERNE-51.xlsx
Processing CBC ERNE-06.xlsx
Processing CBC ERNE-71.xlsx
Processing CBC ERNE-26.xlsx
Processing CBC ERNE-30.xlsx
Processing CBC ERNE-67.xlsx
Processing CBC ERNE-01.xlsx
Processing CBC ERNE-56.xlsx
Processing CBC ERNE-40.xlsx
Processing CBC ERNE-17.xlsx
Processing CBC ERNE-60.xlsx
Processing CBC ERNE-37.xlsx
Processing CBC ERNE-21.xlsx
Processing CBC ERNE-20.xlsx
Processing CBC ERNE-36.xlsx
Processing CBC ERNE-61.xlsx
Processing CBC ERNE-16.xlsx
Processing CBC ERNE-41.xlsx
Processing CBC ERNE-57.xlsx
Processing CBC ERNE-15.xlsx
Processing CBC ERNE-42.xlsx
Processing CBC ERNE-54.xlsx
Processing CBC ERNE-03.xlsx
Processing CBC ERNE-39.xlsx
Processing CBC ERNE-19.xlsx
Processing CBC ERNE-

/var/folders/29/32_nc36n5_14k43tc7g6b_l80000gn/T/ipykernel_32922/3501882348.py:147: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(all_data, ignore_index=True)



✅ Debug file saved: CBC_combined_debug.xlsx
Shape: (351, 33)
Plotting record_id
Plotting wbc
Plotting lym
Plotting mon
Plotting neu
Plotting eos
Plotting bas
Plotting lym_percent
Plotting mon_percent
Plotting neu_percent
Plotting eos_percent
Plotting bas_percent
Plotting rbc
Plotting hgb
Plotting hct
Plotting mcv
Plotting mch
Plotting mchc
Plotting rdwc
Plotting rdws
Plotting plt
Plotting mpv
Plotting pct
Plotting pdwc
Plotting pdws
Plotting form_1_complete
✅ DONE: CBC processing + plotting complete.


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# FILES
# ==========================================================

input_dir = "."   # directory with CBC excel files
tracker_file = "Experiment Tracker.xlsx"

output_dir = "CBC_Plots"
os.makedirs(output_dir, exist_ok=True)

output_image = os.path.join(output_dir, "CBC_Multimetric_Panel.png")

# ==========================================================
# GLOBAL PLOT CONFIG
# ==========================================================

PLOT_CONFIG = {
    "figsize": (40, 15),
    "dpi": 300,

    "axis_tick_size": 18,
    "label_size": 20,
    "title_size": 28,
    "legend_size": 16,

    "x_tick_interval": 30,
    "x_min": 0,
    "x_max": 240,

    "line_width": 2,
    "marker_size": 8,
    "fill_alpha": 0.2,
}

# Optional: make matplotlib text bold by default
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.axisbelow"] = True

# ==========================================================
# TIME CONTROL
# ==========================================================

order_of_time_points = [
    0, 30, 60, 85, 120, 240
]

# ==========================================================
# CBC METRICS TO PLOT AS PANEL ROWS
# Use exact Excel column names here
# ==========================================================

metrics_to_plot = [
    "wbc",
    "rbc",
    "plt"
]

row_label_map = {
    "wbc": "(A) WBC",
    "rbc": "(B) RBC",
    "plt": "(C) Platelets"
}

metric_name_map = {
    "wbc": "WBC (10³/µL)",
    "rbc": "RBC (10⁶/µL)",
    "hgb": "Hemoglobin (g/dL)",
    "hct": "Hematocrit (%)",
    "plt": "Platelets (10³/µL)"
}

# ==========================================================
# STYLE DEFINITIONS
# ==========================================================

marker_info = {
    "No Occlusion":      {"marker": "o", "color": "black"},
    "Partial Occlusion": {"marker": "o", "color": "blue"},
    "Full Occlusion":    {"marker": "o", "color": "red"}
}

occlusion_order = [
    "No Occlusion",
    "Partial Occlusion",
    "Full Occlusion"
]

occlusion_label_map = {
    "No Occlusion": "No REBOA",
    "Partial Occlusion": "p-REBOA",
    "Full Occlusion": "f-REBOA"
}

# ==========================================================
# PARSE NAME COLUMN
# ==========================================================

def parse_name(value):
    try:
        parts = str(value).split("-")

        # Convert ERNE-61 → ERNE61
        subject = parts[0] + parts[1]

        time_point = parts[2]

        return subject, time_point

    except:
        return None, None

def convert_time(tp):
    if tp == "PRE":
        return 0

    if isinstance(tp, str) and tp.startswith("T"):
        try:
            return float(tp.replace("T", ""))
        except:
            return None

    return None

# ==========================================================
# LOAD TRACKER
# ==========================================================

tracker = pd.read_excel(
    tracker_file,
    sheet_name=["T30 categorization", "T60 categorization"]
)

group_10 = set(tracker["T30 categorization"]["Group_10"].dropna().astype(str))
group_20 = set(tracker["T30 categorization"]["Group_20"].dropna().astype(str))
group_30 = set(tracker["T30 categorization"]["Group_30"].dropna().astype(str))

full_occlusion = set(tracker["T60 categorization"]["Full_Occlusion"].dropna().astype(str))
partial_occlusion = set(tracker["T60 categorization"]["Partial_Occlusion"].dropna().astype(str))
no_occlusion = set(tracker["T60 categorization"]["No_Occlusion"].dropna().astype(str))

# ==========================================================
# LOAD + CLEAN ALL CBC FILES
# ==========================================================

all_data = []

for file in os.listdir(input_dir):
    if file.endswith(".xlsx") and not file.startswith("~$") and file != tracker_file:

        print(f"Processing {file}")

        temp_df = pd.read_excel(os.path.join(input_dir, file))

        if "name" not in temp_df.columns:
            continue

        # Split name column
        parsed = temp_df["name"].apply(parse_name)
        temp_df[["Subject", "Time Point"]] = pd.DataFrame(
            parsed.tolist(),
            index=temp_df.index
        )

        # Convert time
        temp_df["Time Point"] = temp_df["Time Point"].apply(convert_time)

        # Drop bad rows
        temp_df = temp_df.dropna(subset=["Subject", "Time Point"])

        all_data.append(temp_df)

if len(all_data) == 0:
    raise ValueError("No CBC Excel files with a 'name' column were found.")

# Combine all files
df = pd.concat(all_data, ignore_index=True)

# ==========================================================
# APPLY TRACKER LOGIC
# ==========================================================

def assign_hemorrhage(subj):
    if subj in group_10:
        return 10
    elif subj in group_20:
        return 20
    elif subj in group_30:
        return 30

    return np.nan

def assign_occlusion(subj):
    if subj in full_occlusion:
        return "Full Occlusion"
    elif subj in partial_occlusion:
        return "Partial Occlusion"
    elif subj in no_occlusion:
        return "No Occlusion"

    return np.nan

df["Hemorrhage Level"] = df["Subject"].apply(assign_hemorrhage)
df["Occlusion Group"] = df["Subject"].apply(assign_occlusion)

# Remove subjects not in tracker
df = df.dropna(subset=["Hemorrhage Level", "Occlusion Group"])

# ==========================================================
# FILTER TIME POINTS
# ==========================================================

df["Time Point"] = pd.to_numeric(df["Time Point"], errors="coerce")
df = df[df["Time Point"].isin(order_of_time_points)]
df = df.sort_values("Time Point")

# ==========================================================
# EXPORT FINAL CLEANED DATA FOR DEBUGGING
# ==========================================================

debug_output_file = "CBC_combined_debug.xlsx"

df.to_excel(debug_output_file, index=False)

print(f"\n✅ Debug file saved: {debug_output_file}")
print("Shape:", df.shape)

# ==========================================================
# MAKE SURE REQUESTED METRICS EXIST
# ==========================================================

numeric_columns = df.select_dtypes(include=["number"]).columns
valid_metrics = [m for m in metrics_to_plot if m in numeric_columns]

if len(valid_metrics) == 0:
    raise ValueError(
        "None of the requested CBC metrics were found in the numeric Excel columns.\n"
        f"Requested metrics: {metrics_to_plot}\n"
        f"Available numeric columns: {list(numeric_columns)}"
    )

missing_metrics = [m for m in metrics_to_plot if m not in numeric_columns]

if len(missing_metrics) > 0:
    print("Warning: These requested CBC metrics were not found and will be skipped:")
    print(missing_metrics)
    print("Available numeric columns:")
    print(list(numeric_columns))

# ==========================================================
# PANEL PLOTTING FUNCTION
# ==========================================================

def create_cbc_multimetric_panel():

    hemorrhage_levels = [10, 20, 30]

    n_rows = len(valid_metrics)
    n_cols = len(hemorrhage_levels)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(10 * n_cols, 5 * n_rows),
        sharex=True,
        sharey="row"
    )

    if n_rows == 1:
        axes = np.array([axes])

    handles, labels = [], []

    for row_idx, metric in enumerate(valid_metrics):
        for col_idx, hemorrhage_level in enumerate(hemorrhage_levels):

            ax = axes[row_idx, col_idx]

            level_df = df[df["Hemorrhage Level"] == hemorrhage_level]

            if level_df.empty:
                ax.set_title(
                    f"{hemorrhage_level:g}% Hemorrhage - No Data",
                    fontsize=PLOT_CONFIG["title_size"],
                    fontweight="bold"
                )
                continue

            mean_df = level_df.groupby(
                ["Occlusion Group", "Time Point"],
                as_index=False
            )[metric].mean()

            std_df = level_df.groupby(
                ["Occlusion Group", "Time Point"],
                as_index=False
            )[metric].std()

            for occlusion_key in occlusion_order:

                mean_subset = mean_df[mean_df["Occlusion Group"] == occlusion_key]
                std_subset = std_df[std_df["Occlusion Group"] == occlusion_key]

                if mean_subset.empty:
                    continue

                mean_subset = mean_subset.sort_values("Time Point")
                std_subset = std_subset.sort_values("Time Point")

                color = marker_info[occlusion_key]["color"]
                marker = marker_info[occlusion_key]["marker"]

                legend_label = occlusion_label_map.get(occlusion_key, occlusion_key)

                line, = ax.plot(
                    mean_subset["Time Point"],
                    mean_subset[metric],
                    marker=marker,
                    markersize=PLOT_CONFIG["marker_size"],
                    linewidth=PLOT_CONFIG["line_width"],
                    color=color,
                    label=legend_label
                )

                ax.fill_between(
                    mean_subset["Time Point"],
                    mean_subset[metric] - std_subset[metric],
                    mean_subset[metric] + std_subset[metric],
                    alpha=PLOT_CONFIG["fill_alpha"],
                    color=color
                )

                if legend_label not in labels:
                    handles.append(line)
                    labels.append(legend_label)

            # Column titles
            if row_idx == 0:
                ax.set_title(
                    f"{hemorrhage_level:g}% Hemorrhage",
                    fontsize=PLOT_CONFIG["title_size"],
                    fontweight="bold"
                )

            # Row labels
            if col_idx == 0:
                ax.set_ylabel(
                    row_label_map.get(metric, metric),
                    fontsize=PLOT_CONFIG["label_size"],
                    fontweight="bold"
                )

            # Bottom x-labels
            if row_idx == n_rows - 1:
                ax.set_xlabel(
                    "Time (mins)",
                    fontsize=PLOT_CONFIG["label_size"],
                    fontweight="bold"
                )

            # X ticks
            ax.set_xticks(
                np.arange(
                    PLOT_CONFIG["x_min"],
                    PLOT_CONFIG["x_max"] + 1,
                    PLOT_CONFIG["x_tick_interval"]
                )
            )

            ax.set_xlim(
                PLOT_CONFIG["x_min"],
                PLOT_CONFIG["x_max"]
            )

            ax.tick_params(
                axis="both",
                labelsize=PLOT_CONFIG["axis_tick_size"]
            )

            # Bold tick labels
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                tick.set_fontweight("bold")

            ax.grid(True, linestyle="--", linewidth=0.7)

    # Legend intentionally not plotted
    # fig.legend(
    #     handles,
    #     labels,
    #     loc="center right",
    #     fontsize=PLOT_CONFIG["legend_size"]
    # )

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    plt.savefig(
        output_image,
        dpi=PLOT_CONFIG["dpi"]
    )

    plt.close()

# ==========================================================
# RUN
# ==========================================================

create_cbc_multimetric_panel()

print("✅ DONE: CBC processing + panel plotting complete.")

Processing CBC ERNE-66.xlsx
Processing CBC ERNE-31.xlsx
Processing CBC ERNE-27.xlsx
Processing CBC ERNE-70.xlsx
Processing CBC ERNE-07.xlsx
Processing CBC ERNE-50.xlsx
Processing CBC ERNE-46.xlsx
Processing CBC ERNE-11.xlsx
Processing CBC ERNE-10.xlsx
Processing CBC ERNE-47.xlsx
Processing CBC ERNE-51.xlsx
Processing CBC ERNE-06.xlsx
Processing CBC ERNE-71.xlsx
Processing CBC ERNE-26.xlsx
Processing CBC ERNE-30.xlsx
Processing CBC ERNE-67.xlsx
Processing CBC ERNE-01.xlsx
Processing CBC ERNE-56.xlsx
Processing CBC ERNE-40.xlsx
Processing CBC ERNE-17.xlsx
Processing CBC ERNE-60.xlsx
Processing CBC ERNE-37.xlsx
Processing CBC ERNE-21.xlsx
Processing CBC ERNE-20.xlsx
Processing CBC ERNE-36.xlsx
Processing CBC ERNE-61.xlsx
Processing CBC ERNE-16.xlsx
Processing CBC ERNE-41.xlsx
Processing CBC ERNE-57.xlsx
Processing CBC ERNE-15.xlsx
Processing CBC ERNE-42.xlsx
Processing CBC ERNE-54.xlsx
Processing CBC ERNE-03.xlsx
Processing CBC ERNE-39.xlsx
Processing CBC ERNE-19.xlsx
Processing CBC ERNE-

/var/folders/29/32_nc36n5_14k43tc7g6b_l80000gn/T/ipykernel_32922/662476942.py:183: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(all_data, ignore_index=True)



✅ Debug file saved: CBC_combined_debug.xlsx
Shape: (702, 33)
✅ DONE: CBC processing + panel plotting complete.
